### HyDE
🧠 What is HyDE?

HyDE (Hypothetical Document Embeddings) is a retrieval technique where, instead of embedding the user's query directly, you first generate a hypothetical answer (document) to the query using an LLM, and then embed that hypothetical document to search your vector store.

➡️ HyDE bridges the gap between user intent and relevant content, especially when:

1. Queries are short
2. There is a language mismatch between query and documents
3. You want to retrieve based on answer content, not question words

In [2]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


C:\Users\Adit Sharma\AppData\Local\Temp\ipykernel_4328\888061924.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader


In [3]:
# 1. Load and chunk your dataset
import wikipedia
from datetime import timedelta

# Wikipedia blocks the package's default User-Agent with 429 errors → set your own and slow down
wikipedia.set_user_agent("RAG-learning-notebook/1.0")
wikipedia.set_rate_limiting(True, min_wait=timedelta(seconds=1))

chunk_size = 300
chunk_overlap = 100

# loading data
loader = WikipediaLoader(query="Steve Jobs", load_max_docs=5)
documents = loader.load()

# text splitting
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
docs = text_splitter.split_documents(documents=documents)
print(len(docs))
docs[:3]


84


[Document(metadata={'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. with his early business partner Steve Wozniak as Apple Computer Company in 1976. After the company\'s board of directors fired him in 1985, he founded NeXT the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation in 2011.\nJobs was born in San Francisco in 1955 and adopted shortly afterward. He attended Reed College in 1972 before withdrawing that same year. In 1974, he traveled through India, seeking enlightenment before later studying Zen Buddhism. He and Wozniak co-founded Apple in 1976 to further develop and s

In [4]:
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("groq_api")
llm = init_chat_model("groq:openai/gpt-oss-20b")

In [6]:
db=Chroma.from_documents(documents=docs,embedding=embeddings,persist_directory="output/steve_jobs_for_hyde.db")
base_retriever = db.as_retriever(search_kwargs={"k": 5})

In [7]:
def get_hyde_doc(query):
    template = """Imagine you are an expert writing a detailed explanation on the topic: '{query}'
    create a hypothetical answer for the topic"""

    system_message_prompt = SystemMessagePromptTemplate.from_template(template=template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_prompt(query=query).to_messages()
    response = llm.invoke(messages)
    hypo_doc = response.content
    return hypo_doc

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.chat import SystemMessagePromptTemplate
query = "When was Steve Jobs fired from Apple?"
hypo_doc = get_hyde_doc(query=query)
print(hypo_doc)


**When was Steve Jobs fired from Apple? – A Detailed Exploration**

---

### 1.  The Road to 1985: A Quick Recap

| Year | Key Milestone |
|------|---------------|
| **1976** | Steve Jobs, Steve Wozniak, and Ronald Wayne founded Apple Computer, Inc. |
| **1977** | Apple launched the Apple II, becoming a commercial success. |
| **1981** | Apple went public; Jobs became the youngest CEO in the world. |
| **1984** | Apple introduced the Macintosh, the first mass‑market computer with a graphical user interface. |
| **1985** | The infamous board meeting that led to Jobs’ departure. |

By 1985, Apple was facing a critical crossroads. The Macintosh, despite its visionary design, had not delivered the sales or profit margins the board expected. At the same time, a power struggle had erupted between Jobs and the company’s founder‑CEO, **John Sculley**, who had been hired in 1983 as a “big‑business” executive.

---

### 2.  The Board Meeting – March 1985

**Date:** *March 13, 1985*  
**Location:

In [10]:
print("🔎 Normal retrieval (raw query):")
for d in base_retriever.invoke(query):
    print("-", d.page_content[:120])

print("\n🧪 HyDE retrieval (hypothetical doc):")
matched_doc = base_retriever.invoke(hypo_doc)
for d in matched_doc:
    print("-", d.page_content[:120])


🔎 Normal retrieval (raw query):
- 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was clo
- In 1985, Jobs departed Apple after a long power struggle with the company's board and its then-CEO, John Sculley. That s
- Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer o
- Apple CEO John Sculley demands to know why the world believes he fired Jobs – Jobs was actually forced out by the Apple 
- In 1997, Jobs returned to Apple as CEO after the company acquired NeXT. He was largely responsible for reviving Apple, w

🧪 HyDE retrieval (hypothetical doc):
- In 1985, Jobs departed Apple after a long power struggle with the company's board and its then-CEO, John Sculley. That s
- Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer o
- 1986, becoming its chairman and majority shareholder until 2007. Jo

### Langchain-HypotheticalDocumentEmbedder

In [12]:
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Step 1: Load and split documents
loader = TextLoader("langchain_crewai_dataset.txt", encoding="utf-8")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(len(chunks))

241


In [13]:
# Step 2: Set up base embeddings
base_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

According to the LangChain source code (`PROMPT_MAP`), the built-in `prompt_key` options are:

- web_search
- sci_fact
- arguana
- trec_covid
- fiqa
- dbpedia_entity
- trec_news
- mr_tydi


In [14]:
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    prompt_key="web_search"
)


In [15]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=hyde_embedding_function,
    persist_directory="output/langchain"
)


In [16]:
rag_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
rag_chain = create_stuff_documents_chain(llm=llm, prompt=rag_prompt)

In [17]:
def hyde_rag_pipeline(query):
    matched_docs = vectorstore.similarity_search(query, k=4)   # query → LLM passage → embedding → search
    for d in matched_docs:
        print("-", d.page_content[:100])
    response = rag_chain.invoke({
        "input": query,
        "context": matched_docs
    })
    return response

In [18]:
query = "What memory modules does LangChain provide?"
answer = hyde_rag_pipeline(query)
print("\n✅ Final Answer:\n", answer)

- LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These a
- LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These a
- LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These a
- LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These a

✅ Final Answer:
 LangChain provides two primary memory modules:

1. **ConversationBufferMemory** – Keeps a running buffer of the entire conversation history, allowing the LLM to reference all prior turns.

2. **ConversationSummaryMemory** – Stores a concise summary of the conversation, which is useful for keeping the dialogue within token limits while still preserving essential context.


In [ ]:
custom = PromptTemplate.from_template(
    "Generate a concise hypothetical answer for this topic: {query}"
)

hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    custom_prompt=custom
)

# chunk vectors don't change, so reuse the existing store and only re-embed the query
qvec = hyde_embedding_function.embed_query(query)
for d in vectorstore.similarity_search_by_vector(qvec, k=4):
    print("-", d.page_content[:100])


- LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These a
- LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These a
- LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These a
- LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These a


: 